# Notebook16 equal active-wall runner
Inputs: frozen two-arm confirmation plan and original audited Food101 train/development memberships. Outputs: full-state checkpoints under checkpoints/<run_id>/stages/worker*/arms, durable active-wall journals, endpoint receipts. Runtime: bounded by the original12hour parent deadline. Source-bound checkpoints retain latest two and best. Run the frontdoor16 in a fresh kernel with the same RUN_ID to resume; an IN_FLIGHT timing journal refuses scientific continuation with UNKNOWN fairness. No test pixels are read here.

In [ ]:
"""Equal active-wall full-JEPA training with fail-closed durable time accounting."""
from pathlib import Path
import copy,gc,json,math,os,time
import nbformat,torch
_SELF_PATH=Path(globals().get('__file__','/home/benjamin/workspace/ahr_go/fmm_jeppa/experiments/16_hierarchical_confirmation/runner.ipynb'))
_COMMON_PATH=_SELF_PATH.with_name('common.ipynb');_namespace={'__file__':str(_COMMON_PATH)}
for _cell in nbformat.read(_COMMON_PATH,4).cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,str(_COMMON_PATH),'exec'),_namespace)
C=type('Common',(),_namespace)
RESUME_NOTEBOOK='notebooks/16_Hierarchical_Scaling_and_Confirmation.ipynb'
RESUME_CELL='launch-hierarchical-confirmation'

def bind(root,hierarchy):
    runner,sources=C.bind_backend(root,hierarchy)
    runner.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner.RESUME_CELL=RESUME_CELL
    runner._BASE.RESUME_NOTEBOOK=RESUME_NOTEBOOK;runner._BASE.RESUME_CELL=RESUME_CELL
    return runner,sources

def make_plan(root,run_id,proposal,handoff,gate_receipts,hierarchy,image_size=256,devices=('cuda:0','cuda:1')):
    root=Path(root);base=C.read(handoff['candidate_plan'])['arms'][0]['config'];sources=C.source_hashes(root)
    evaluation=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm16_plan_evaluation');probe=evaluation.probe_configuration(base.get('probe_convex'))
    seeds=list(map(int,proposal['seeds']));updates=int(proposal['schedule_updates']);arms=[]
    for number,seed in enumerate(seeds):
        for variant in ('vit','optional_geometry'):
            cfg=copy.deepcopy(base);cfg['image_size']=int(image_size);cfg['device']=devices[number%len(devices)];cfg['dtype']='float32';cfg['cpu_threads']=2
            cfg['seeds']['model_init']=seed;cfg['seeds']['data_order']=seed+101;cfg['seeds']['augmentation']=seed+201;cfg['seeds']['mask']=seed+301
            cfg['training'].update(updates=updates,microbatch=32,accumulation_steps=4,checkpoint_every=50)
            cfg['budget'].update(max_seconds=43200,target_seconds=43200,round_updates=50)
            cfg.setdefault('evaluation',{})['heldout_enabled']=False
            cfg['kernel15']['variant']=variant;cfg.pop('kernel14',None);cfg['probe_convex']=copy.deepcopy(probe)
            arms.append({'id':'s'+str(seed)+'-'+variant,'seed':seed,'variant':variant,'condition':variant,'kind':'vit'if variant=='vit'else'semantic','config':cfg})
    main=root/RESUME_NOTEBOOK
    code=[''.join(c['source'])for c in json.loads(main.read_text())['cells']if c['cell_type']=='code']
    return {'schema':'notebook16-equal-active-wall-plan/v1','run_id':run_id,'stage_id':'confirmation','seeds':seeds,'registered_seeds':seeds,
        'variants':['vit','optional_geometry'],'arms':arms,'updates':updates,'milestones':[updates],'round_updates':50,'effective_batch':128,
        'probe_endpoints':['final'],'probe_convex':probe,'train_count':10100,'probe_train_count':10100,'hard_limit_seconds':43200,
        'active_wall_seconds_per_arm':float(proposal['active_wall_seconds_per_arm']),'wall_tolerance_fraction':.02,'wall_tolerance_seconds':2.,
        'active_wall_scope':'Every timed training invocation, including preprocessing, synchronization, checkpoint/metrics writes and model load/cleanup. Inter-invocation process downtime excluded only after committed READY journal.',
        'crash_policy':'IN_FLIGHT journal on restart fails UNKNOWN fairness; optimizer checkpoint remains exactly resumable but no unmeasured wall time is silently forgiven.',
        'hierarchy':C.canonical(hierarchy),'source_hashes':sources,'gate_receipts':gate_receipts,
        'resume_notebook':RESUME_NOTEBOOK,'resume_cell':RESUME_CELL,'consumer_notebook_sha256':C.digest(code),'official_test':'CLOSED'}

def worker_plan(plan,worker):
    child=copy.deepcopy(plan);index=int(worker.replace('worker',''));seeds=plan['seeds'][index::2]
    child['stage_id']=worker;child['seeds']=seeds;child['arms']=[a for a in plan['arms']if int(a['seed'])in seeds]
    return child

def checkpoint_record(runner,directory):
    manifest=runner.validate_training_manifest(directory);path=Path(directory)/manifest['latest'];record=C.file_record(path)
    return manifest,record

def timer_binding(plan,arm):
    return {'plan_sha256':C.digest(plan),'arm_sha256':C.digest(arm),'source_hashes':plan['source_hashes'],'budget_seconds':float(plan['active_wall_seconds_per_arm'])}

def validate_timer(timer,binding,manifest=None,record=None):
    if timer['binding']!=binding:raise ValueError('Registered equal-wall timer changed')
    if timer['state']=='IN_FLIGHT':raise RuntimeError('UNKNOWN active-wall fairness: invocation interrupted before atomic timer commitment; no time reset allowed')
    if timer['state']not in('READY','COMPLETE','UNKNOWN'):raise ValueError('Invalid timer state')
    if not math.isfinite(timer['active_seconds'])or timer['active_seconds']<0:raise ValueError('Invalid accumulated active time')
    if timer.get('checkpoint'):
        if record!=timer['checkpoint']or manifest['successful_updates']!=timer['successful_updates']:raise ValueError('Timer/model publication differs; UNKNOWN fairness, never restart')
    elif manifest is not None:raise ValueError('Unaccounted checkpoint exists before timed training')
    return timer

def train_arm(root,artifact,plan,bundle,arm,deadline_unix,max_invocations=None):
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['hierarchy'])
    if sources!=plan['source_hashes']:raise ValueError('Frozen16 sources changed')
    C.environment(arm['config']);folder,directory=runner._arm_paths(root,artifact,plan,arm)
    timer_path=directory/'active_wall_timer.json';binding=timer_binding(plan,arm);directory.mkdir(parents=True,exist_ok=True)
    timer=C.read(timer_path)if timer_path.exists()else {'schema':'notebook16-active-wall-journal/v1','binding':binding,'state':'READY','active_seconds':0.,'successful_updates':0,'checkpoint':None,'invocations':[],'fair_wall_budget_verified':False}
    manifest,record=checkpoint_record(runner,directory)if(directory/'manifest.json').exists()else(None,None)
    validate_timer(timer,binding,manifest,record)
    if not timer_path.exists():C.atomic_json(timer_path,timer)
    budget=float(binding['budget_seconds']);tolerance=max(float(plan['wall_tolerance_seconds']),budget*float(plan['wall_tolerance_fraction']))
    if timer['state']=='COMPLETE':return final_arm(runner,root,artifact,plan,arm,timer)
    if timer['state']=='UNKNOWN':return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False,'timer':C.file_record(timer_path)}
    protocol=C.read(folder/'protocol.json')
    while timer['active_seconds']<budget:
        C.check_deadline(deadline_unix,5.)
        remaining=budget-timer['active_seconds'];before=int(timer['successful_updates'])
        if before>=int(plan['updates']):
            timer.update(state='UNKNOWN',reason='Scheduler update cap reached before registered active-wall budget; time parity unverified');C.atomic_json(timer_path,timer)
            return {'status':'UNKNOWN','arm_id':arm['id'],'reason':timer['reason'],'fair_wall_budget_verified':False}
        invocation_deadline=min(float(deadline_unix),time.time()+remaining)
        marker={'sequence':len(timer['invocations'])+1,'started_unix':time.time(),'before_step':before,'before_checkpoint':timer['checkpoint'],'remaining_seconds':remaining}
        timer.update(state='IN_FLIGHT',in_flight=marker);C.atomic_json(timer_path,timer)
        started=time.perf_counter()
        try:
            result=runner._run_stage_model(plan,arm['kind'],arm['config'],bundle,folder,directory,protocol,
                model=runner._initial_model(root,artifact,plan,arm),stop_after=min(int(plan['updates']),before+int(plan['round_updates'])),
                deadline_unix=invocation_deadline)
            manifest,record=checkpoint_record(runner,directory)
            if manifest['successful_updates']!=result['resume']['final_step']or manifest['attempted_updates']!=manifest['successful_updates']:raise ValueError('Nonfinite or failed updates invalidate timed confirmation')
            performance=copy.deepcopy(result['performance']);del result;gc.collect()
            elapsed=time.perf_counter()-started
            marker.update(ended_unix=time.time(),active_seconds=elapsed,after_step=manifest['successful_updates'],after_checkpoint=record,
                performance=performance)
            timer['active_seconds']+=elapsed;timer['successful_updates']=manifest['successful_updates'];timer['checkpoint']=record;timer['invocations'].append(marker)
            timer.pop('in_flight',None);timer['state']='READY';C.atomic_json(timer_path,timer)
            
        except BaseException:
            # Keep IN_FLIGHT explicitly. A full model checkpoint may be valid, but active wall between publications cannot be recovered honestly.
            raise
        if timer['active_seconds']>=budget:break
        if max_invocations is not None and len(timer['invocations'])>=int(max_invocations):
            return {'status':'READY_FOR_EXACT_RESUME','arm_id':arm['id'],'successful_updates':timer['successful_updates'],'timer':C.file_record(timer_path),'fair_wall_budget_verified':False}
    fair=budget<=timer['active_seconds']<=budget+tolerance and timer['successful_updates']<int(plan['updates'])
    timer.update(state='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,overshoot_seconds=max(0.,timer['active_seconds']-budget),
        reason='Registered equal active-wall budget reached'if fair else'Active-wall overshoot/update cap exceeded registered tolerance')
    C.atomic_json(timer_path,timer);return final_arm(runner,root,artifact,plan,arm,timer)

def final_arm(runner,root,artifact,plan,arm,timer):
    folder,directory=runner._arm_paths(root,artifact,plan,arm);manifest,record=checkpoint_record(runner,directory)
    validate_timer(timer,timer_binding(plan,arm),manifest,record)
    return {'status':'COMPLETE'if timer['fair_wall_budget_verified']else'UNKNOWN','arm_id':arm['id'],'seed':arm['seed'],'variant':arm['variant'],'model_id':arm['kind'],
        'actual_updates':manifest['successful_updates'],'attempted_updates':manifest['attempted_updates'],'image_exposures':manifest['image_exposures'],
        'latest_checkpoint':record,'source_encoder':record,'initial_teacher':C.read(artifact/'stage_registration.json')['arms'][arm['id']]['initial_teacher'],
        'fair_wall_budget_verified':timer['fair_wall_budget_verified'],'active_wall_seconds':timer['active_seconds'],'timer':C.file_record(directory/'active_wall_timer.json'),
        'schedule_updates':plan['updates'],'selection':'Last registered wall-budget checkpoint; never best-dev/epoch selected'}

def train_worker(root,artifact,plan,bundle,deadline_unix):
    from fmm_jepa.food101_training import run_writer
    root,artifact=Path(root),Path(artifact);runner,sources=bind(root,plan['hierarchy']);C.environment(plan['arms'][0]['config'])
    if len(plan['arms'])*float(plan['active_wall_seconds_per_arm'])>900 and not os.environ.get('TMUX'):raise RuntimeError('Long16 worker must run in detached tmux through the frontdoor')
    if sources!=plan['source_hashes']:raise ValueError('16 source binding changed')
    runner.prepare_stage(root,artifact,plan,bundle);preflight=runner.verify_stage_resume(root,artifact,plan,bundle,deadline_unix)
    if preflight['status']!='PASS':raise ValueError('Actual16 full-state exact-resume preflight failed')
    path=artifact/'training_result.json';saved=C.read(path)if path.exists()else {'status':'RUNNING','plan_hash':C.digest(plan),'arms':{},'fair_wall_budget_verified':False,'gate_receipts':plan['gate_receipts']}
    if saved['plan_hash']!=C.digest(plan):raise ValueError('Timed worker plan changed')
    with run_writer(artifact/'timed_training_writer'):
        for arm in plan['arms']:
            saved['arms'][arm['id']]=train_arm(root,artifact,plan,bundle,arm,deadline_unix);C.atomic_json(path,saved)
            if saved['arms'][arm['id']]['status']!='COMPLETE':saved['status']='UNKNOWN';C.atomic_json(path,saved);return saved
    values=list(saved['arms'].values());tolerance=max(plan['wall_tolerance_seconds'],plan['active_wall_seconds_per_arm']*plan['wall_tolerance_fraction'])
    fair=all(v['fair_wall_budget_verified']for v in values)and max(v['active_wall_seconds']for v in values)-min(v['active_wall_seconds']for v in values)<=tolerance
    saved.update(status='COMPLETE'if fair else'UNKNOWN',fair_wall_budget_verified=fair,preflight=C.file_record(artifact/'stage_preflight.json'))
    C.atomic_json(path,saved);return saved

def evaluate_worker(root,artifact,plan,bundle,training,deadline_unix):
    root,artifact=Path(root),Path(artifact);module=C.load(root,'experiments/15_vit_geometry/evaluation.ipynb','fmm16_timed_evaluation')
    module.bind_model_builder(C.builder(root,plan['hierarchy']),plan['source_hashes']);rows=[]
    for arm in plan['arms']:
        saved=training['arms'][arm['id']]
        if saved['status']!='COMPLETE':return {'status':'UNKNOWN','reason':'Incomplete timed training; test remains closed'}
        row=module.evaluate_endpoint(root,artifact/'evaluation',arm,bundle,saved['source_encoder'],saved['actual_updates'],deadline_unix,
            probe_config=plan['probe_convex'],endpoint='final',training_cost={'active_wall_seconds':saved['active_wall_seconds']},run_id=plan['run_id'],stage_id=plan['stage_id'])
        rows.append(row);C.atomic_json(artifact/'evaluation_result.json',{'status':'RUNNING','rows':rows,'official_test':'CLOSED'})
        if not row.get('converged'):
            result={'status':'UNKNOWN','rows':rows,'reason':'Readout convergence not certified','official_test':'CLOSED'};C.atomic_json(artifact/'evaluation_result.json',result);return result
    result={'status':'COMPLETE','rows':rows,'official_test':'CLOSED'};C.atomic_json(artifact/'evaluation_result.json',result);return result
